# RNN Models
Composed of 3 models (one trained on all data, one on critics, one on users)

## Setup
Ensure that pytorch is installed first, as well as any other dependencies

In [2]:
import torch
print(torch.cuda.is_available())   

False


In [ ]:
import copy
import io
import random
import re
import shutil
import time
import unicodedata
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Selected device:", DEVICE)

PAD, UNK = 0, 1
MAX_SENTENCE_TOKENS = 9
MAX_PAIRS = 3000
BATCH_SIZE = 64
EPOCHS=50

Selected device: cpu


### Helper Functions

In [ ]:
def label_to_score(label):
    return label * 10

def score_to_label(score):
    return score // 10

@torch.no_grad
def predict_score(review, model, vocab, device=DEVICE):
    model.eval()
    tokens = vocab.encode(review)
    src = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)
    src_lengths = torch.tensor([len(src)], dtype=torch.long).to(device)
    logits = model(src, src_lengths)
    label = logits.argmax(-1).item()
    return label_to_score(label)

## Vocabulary and Minibatches

### Vocab

In [ ]:
class Vocab:
    def __init__(self):
        self.word2id = {"<pad>": PAD, "<unk>": UNK}
        self.id2word = {PAD: "<pad>", UNK: "<unk>"}
    
    def __len__(self):
        return len(self.word2id)
    
    def add_sentence(self, sentence):
        for word in sentence.split():
            if word not in self.word2id:
                id = len(self.word2id)
                self.word2id[word] = id
                self.id2word[id] = word

    def encode(self, sentence):
        return [self.word2id.get(word, UNK) for word in sentence.split()]

### Dataset

In [ ]:
class MovieReviewDataset(Dataset):
    def __init__(self, reviews, scores, vocab):
        self.reviews = reviews
        self.scores = scores
        self.vocab = vocab
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, index):
        text = self.vocab.encode(self.reviews[index])
        label = score_to_label(self.scores[index])
        return text, label

### Collate

In [ ]:
def collate(batch):
    src_sequences = []
    src_lengths = []
    labels = []
    for src, label in batch:
        src_sequences.append(torch.tensor(src, dtype=torch.long))
        src_lengths.append(len(src))
        labels.append(label)
    src = pad_sequence(src_sequences, batch_first=True, padding_value=PAD)
    src_lengths = torch.tensor(src_lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return src, src_lengths, labels

## RNN Model

### Cell

In [4]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_linear = nn.Linear(input_size, hidden_size)
        self.hidden_linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, x_t, h_prev):
        input = self.input_linear(x_t)
        hidden = self.hidden_linear(h_prev)
        state = input + hidden
        return torch.tanh(state)

### Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, source_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(
            source_vocab_size, embedding_dim, padding_idx=PAD
        )
        self.dropout = nn.Dropout(dropout)
        self.cell = RNNCell(input_size=embedding_dim, hidden_size=hidden_size)

    def forward(self, src, src_length):
        embedded = self.dropout(self.embedding(src))
        batch_size = src.size(0)
        h = embedded.new_zeros(batch_size, self.hidden_size)
        for t in range(src.size(1)):
            proposal = self.cell(embedded[:, t], h)
            mask = (t < src_length).unsqueeze(1)
            h = torch.where(mask, proposal, h)
        return h

### Score Classifier

In [ ]:
class ScoreClassifier(nn.Module):
    def __init__(self, source_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1, num_classes=11):
        super().__init__()
        self.encoder = Encoder(source_vocab_size, embedding_dim, hidden_size, dropout)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, src, src_lengths):
        encoder = self.encoder(src, src_lengths)
        logits = self.classifier(encoder)
        return logits

## Training and Validation

### Training One Epoch

In [ ]:
def train_one_epoch(model, loader, criterion,
                    optimizer, device, clip_value=1.0):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    for src, src_lengths, labels in loader:
        src, src_lengths, labels = src.to(device), src_lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(src, src_lengths)
        loss = criterion(logits, labels)
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimizer.step()
        predictions = logits.argmax(dim=-1)
        total_loss += loss.item() * src.size(0)
        total_correct += int((predictions == labels).sum().item())
        total_tokens += src.size(0)
    return total_loss / total_tokens, total_correct / total_tokens

### Evaluate

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_tokens = 0
    for src, src_lengths, labels in loader:
        src, src_lengths, labels = src.to(device), src_lengths.to(device), labels.to(device)
        logits = model(src, src_lengths)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=-1)
        total_loss += float(loss.item()) * src.size(0)
        total_correct += int((predictions == labels).sum().item())
        total_tokens += src.size(0)
    return total_loss / total_tokens, total_correct / total_tokens

### Fit Model

In [ ]:
def fit_model(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, 
              weight_decay=1e-4, clip_value=1.0, device=DEVICE, verbose=True):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion,
                                          optimizer, device, clip_value)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        for key, value in (("train_loss", tr_loss), ("train_acc", tr_acc),
                           ("val_loss", va_loss), ("val_acc", va_acc)):
            history[key].append(value)
        if va_loss < best_loss:
            best_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"Epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} | "
                  f"val loss {va_loss:.4f} | train acc {tr_acc:.1%} | val acc {va_acc:.1%}")
    model.load_state_dict(best_state)
    return history

## Executables

### RNN (CRITICS + USERS)

In [ ]:
model = ScoreClassifier(source_vocab_size=len(vocab), embedding_dim=64, hidden_size=64)
history = fit_model(model, train_loader, val_loader, epochs=EPOCHS)

## Insert Graphs and Other here